In [2]:
import matplotlib.pyplot as plt
import pandas as pd

data = pd.read_csv(
    'rmsf_res.xvg',
    comment='@',
    delim_whitespace=True,
    header=None,
    names=['Time', 'RMSF'],
)

# Basic statistical analysis
mean_pe = data['RMSF'].mean()
print(f'Average RMSF: {mean_pe:.2f}')

# Plotting the energy trajectory
plt.plot(data['Time'], data['RMSF'])
plt.xlabel('Time (ps)')
plt.ylabel('RMSF')
plt.title('RMSF over Time')
plt.show()


/tmp/ipykernel_55040/391043776.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(


ParserError: Error tokenizing data. C error: Expected 10 fields in line 10, saw 13


In [ ]:
import numpy as np

def read_xvg(path):
    """Read a GROMACS .xvg file. Returns (data, meta)."""
    meta = {"title": "", "xaxis": "", "yaxis": "", "legends": []}
    rows = []
    with open(path) as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            if line.startswith("@"):
                if "xaxis" in line and '"' in line:
                    meta["xaxis"] = line.split('"')[1]
                elif "yaxis" in line and '"' in line:
                    meta["yaxis"] = line.split('"')[1]
                elif line.startswith("@ s") and "legend" in line:
                    meta["legends"].append(line.split('"')[1])
            continue
            rows.append([float(v) for v in line.split()])
    return np.array(rows), meta

In [ ]:
read_xvg('rmsf_res.xvg')

In [ ]:
import matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.analysis import rms

# 1. Load your simulation files
# Replace with your actual topology (gro, pdb) and trajectory (xtc, dcd, netcdf)
u = mda.Universe('topology.gro', 'trajectory.xtc')
ref = mda.Universe('topology.gro')  # Reference structure (usually first frame)

# Select the atoms to analyze (typically backbone or alpha carbons)
alignment_selection = 'name CA'

# =====================================================================
# 2. Calculate RMSD
# =====================================================================
R = rms.RMSD(
    u,
    ref,
    select=alignment_selection,  # Align on CA to remove global translation/rotation
    groupselections=['protein'],  # Calculate RMSD for the whole protein
)
R.run()

rmsd_data = R.results.rmsd  # Contains: [Frame, Time (ps), RMSD_backbone, RMSD_protein]

# =====================================================================
# 3. Calculate RMSF
# =====================================================================
# Select the atoms you want to analyze fluctuation for (e.g., C-alpha atoms)
calphas = u.select_atoms('name CA')
rms_fluctuation = rms.RMSF(calphas).run()

rmsf_data = rms_fluctuation.results.rmsf

# =====================================================================
# 4. Plotting the Results
# =====================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot RMSD over Time
ax1.plot(rmsd_data[:, 1] / 1000, rmsd_data[:, 3], color='crimson', label='Protein')
ax1.set_xlabel('Time (ns)')
ax1.set_ylabel('RMSD (Å)')
ax1.set_title('Structural Deviation (RMSD) over Time')
ax1.grid(True, linestyle='--')

# Plot RMSF per Residue
residue_numbers = calphas.resnums
ax2.plot(residue_numbers, rmsf_data, color='royalblue', lw=1.5)
ax2.set_xlabel('Residue Number')
ax2.set_ylabel('RMSF (Å)')
ax2.set_title('Local Flexibility (RMSF) per Residue')
ax2.grid(True, linestyle='--')

plt.tight_layout()
plt.show()
